In [7]:
# ============================
# Full block: install solvers + optimal Rangers lineups vs RHP / LHP
# ============================


# --- 1. Imports ---
import pandas as pd
from pyomo.environ import (
    ConcreteModel, Set, Param, Var, NonNegativeReals, Binary,
    Objective, Constraint, maximize, SolverFactory, value
)

# --- 2. Pick an available solver ---
solver_name = None
for cand in ["appsi_highs", "highs", "cbc", "glpk"]:
    try:
        s = SolverFactory(cand)
        if s is not None and s.available(False):
            solver_name = cand
            break
    except Exception:
        continue

if solver_name is None:
    raise RuntimeError("No MILP solver found")

print(f"Using solver: {solver_name}")
solver = SolverFactory(solver_name)

# =========
# 3. LOAD DATA
# =========

excel_file = "rangers_lineup_data_new.xlsx"
sheet_name = "Rangers Stats"

df = pd.read_excel(excel_file, sheet_name=sheet_name)

df["Primary Pos"] = df["Primary Pos"].astype(str).str.strip()
df["Secondary Pos"] = df["Secondary Pos"].astype(str).str.strip()
df["Name"] = df["Name"].astype(str).str.strip()
df["Bats"] = df["Bats"].astype(str).str.strip()

df["wOBA"] = df["wOBA"].astype(float)
df["wOBA_R"] = df["wOBA_vs_RHP"].astype(float)
df["wOBA_L"] = df["wOBA_vs_LHP"].astype(float)

df["BB%"] = df["BB%"].astype(str).str.replace('%', '').astype(float) / 100

players = df["Name"].tolist()

# Allow DH as a position so bat-only guys can play
F = ["C", "1B", "2B", "3B", "SS", "LF", "CF", "RF", "DH"]

Using solver: appsi_highs


In [5]:
df.head()

,Name,Team,G,PA,HR,R,RBI,SB,BB%,K%,...,Off,Def,Def/162,WAR,Primary Pos,Secondary Pos,Bats,Throws,wOBA_R,wOBA_L
0,Corey Seager,TEX,344,1514,84,217,220,6,0.106,17.9%,...,82.5,14.4,6.781395,15.0,SS,1B,L,R,0.409,0.329
1,Marcus Semien,TEX,448,2005,67,285,236,33,0.093,15.4%,...,24.5,34.7,12.547768,12.9,2B,SS,R,R,0.319,0.328
2,Wyatt Langford,TEX,268,1130,38,147,136,41,0.111,23.5%,...,25.4,5.9,3.566418,7.1,LF,CF,R,R,0.316,0.369
3,Nathaniel Lowe,TEX,301,1289,33,151,151,3,0.127,22.5%,...,24.8,-12.4,-6.673754,5.7,1B,DH,L,R,0.349,0.314
4,Adolis García,TEX,437,1816,83,234,267,33,0.076,26.8%,...,7.4,-16.3,-6.042563,5.3,RF,CF,R,R,0.313,0.314


In [6]:
from pyomo.environ import *

# =========
# PARAMETERS
# =========

q = {}
for _, row in df.iterrows():
    name = row["Name"]
    overall = row["wOBA"]
    wR = row["wOBA_R"]
    wL = row["wOBA_L"]

    PA = row["PA"]
    k = 300

    lam = PA / (PA + k)

    q_R = lam * wR + (1 - lam) * overall
    q_L = lam * wL + (1 - lam) * overall


    q[(name, "R")] = q_R
    q[(name, "L")] = q_L


# Lineup weights
raw_w = {
    1: 1.00, 2: 0.97, 3: 0.94, 4: 0.91, 5: 0.88,
    6: 0.85, 7: 0.82, 8: 0.79, 9: 0.76
}
total_raw = sum(raw_w.values())
w = {s: raw_w[s]/total_raw for s in raw_w}

# =========
# DEFENSE (position weighted)
# =========

max_def = df["Def/162"].abs().max()
def_score = {row["Name"]: row["Def/162"] / max_def for _, row in df.iterrows()}

pos_def_weight = {
    "C": 1.35,
    "SS": 1.30,
    "CF": 1.20,
    "2B": 1.10,
    "3B": 1.05,
    "1B": 1.00,
    "LF": 1.00,
    "RF": 1.00,
    "DH": 0.00
}

def_alpha = 0.15

# =========
# ELIGIBILITY + POSITION TYPE MAP
# =========

eligible = {}
pos_type = {}

for _, row in df.iterrows():
    name = row["Name"]
    primary = row["Primary Pos"]
    secondary = row["Secondary Pos"]

    pos_type[name] = (primary, secondary)

    for f in F:
        eligible[(name, f)] = 0

    if primary in F:
        eligible[(name, primary)] = 1
    if isinstance(secondary, str) and secondary in F:
        eligible[(name, secondary)] = 1

# =========
# METRICS
# =========

onbase_score = {}
power_score = {}

for _, row in df.iterrows():
    name = row["Name"]
    onbase_score[name] = 0.7 * row["OBP"] + 0.3 * row["BB%"]
    power_score[name] = 0.7 * row["SLG"] + 0.3 * row["ISO"]

top_onbase = sorted(players, key=lambda x: onbase_score[x], reverse=True)[:5]
top_power = sorted(players, key=lambda x: power_score[x], reverse=True)[:5]
valid_pairs = [(i, j) for i in top_onbase for j in top_power if i != j]

def next_spots(s):
    return [((s) % 9) + 1, ((s+1) % 9) + 1]

# =========
# MODEL
# =========

def solve_lineup(k_hand):

    model = ConcreteModel()

    model.I = Set(initialize=players)
    model.S = Set(initialize=list(range(1,10)))
    model.F = Set(initialize=F)
    model.Pairs = Set(initialize=valid_pairs, dimen=2)

    model.q = Param(model.I, initialize=lambda m,i: q[(i,k_hand)])
    model.w = Param(model.S, initialize=lambda m,s: w[s])
    model.eligible = Param(model.I, model.F, initialize=lambda m,i,f: eligible[(i,f)])

    model.x = Var(model.I, model.S, within=Binary)
    model.y = Var(model.I, model.F, within=Binary)
    model.z = Var(model.Pairs, model.S, within=Binary)

    alpha = 0.05
    sec_penalty = 0.03  # secondary position penalty

    # =========
    # OBJECTIVE
    # =========

    def obj_rule(m):

        # offense
        base = sum(
            m.w[s] * m.q[i] * m.x[i,s]
            for i in m.I for s in m.S
        )

        # synergy
        synergy = sum(
            alpha * onbase_score[i] * power_score[j] * m.z[i,j,s]
            for (i,j) in m.Pairs
            for s in m.S
        )

        # defense (position-weighted)
        defense = sum(
            def_alpha *
            def_score[i] *
            pos_def_weight[f] *
            m.y[i,f]
            for i in m.I for f in m.F
        )

        # secondary position penalty
        penalty = sum(
            sec_penalty *
            m.y[i,f]
            for i in m.I for f in m.F
            if f == pos_type[i][1]   # secondary position
        )

        return base + synergy + defense - penalty

    model.Obj = Objective(rule=obj_rule, sense=maximize)

    # =========
    # CONSTRAINTS
    # =========

    def lineup_complete(m,s):
        return sum(m.x[i,s] for i in m.I) == 1
    model.LineupComplete = Constraint(model.S, rule=lineup_complete)

    def one_spot(m,i):
        return sum(m.x[i,s] for s in m.S) <= 1
    model.OneSpot = Constraint(model.I, rule=one_spot)

    def one_pos(m,f):
        if f == "DH":
            return sum(m.y[i,f] for i in m.I) <= 1
        return sum(m.y[i,f] for i in m.I) == 1
    model.OnePos = Constraint(model.F, rule=one_pos)

    def one_pos_player(m,i):
        return sum(m.y[i,f] for f in m.F) <= 1
    model.OnePosPlayer = Constraint(model.I, rule=one_pos_player)

    def eligibility_rule(m,i,f):
        return m.y[i,f] <= m.eligible[i,f]
    model.Elig = Constraint(model.I, model.F, rule=eligibility_rule)

    def link(m,i):
        return sum(m.x[i,s] for s in m.S) == sum(m.y[i,f] for f in m.F)
    model.Link = Constraint(model.I, rule=link)

    # =========
    # SYNERGY
    # =========

    def z1(m,i,j,s):
        return m.z[i,j,s] <= m.x[i,s]
    model.Z1 = Constraint(model.Pairs, model.S, rule=z1)

    def z2(m,i,j,s):
        return m.z[i,j,s] <= sum(m.x[j,t] for t in next_spots(s))
    model.Z2 = Constraint(model.Pairs, model.S, rule=z2)

    def z3(m,i,j,s):
        return m.z[i,j,s] >= m.x[i,s] + sum(m.x[j,t] for t in next_spots(s)) - 1
    model.Z3 = Constraint(model.Pairs, model.S, rule=z3)

    # =========
    # SOLVE
    # =========

    solver = SolverFactory("highs")
    solver.options["time_limit"] = 60
    solver.options["mip_rel_gap"] = 0.05

    results = solver.solve(model, tee=True)

    print(f"\nTermination vs {k_hand}HP:", results.solver.termination_condition)

    print(f"\nOptimal lineup vs {k_hand}HP:")
    lineup = []
    for s in model.S:
        for i in model.I:
            if value(model.x[i,s]) > 0.5:
                lineup.append((s,i))

    lineup.sort()

    for s,i in lineup:
        pos = [f for f in model.F if value(model.y[i,f]) > 0.5]
        print(f"  Spot {s}: {i} ({pos[0] if pos else 'DH'})")

    print("Objective value:", value(model.Obj))


# =========
# RUN
# =========

solve_lineup("R")
solve_lineup("L")


Termination vs RHP: optimal

Optimal lineup vs RHP:
  Spot 1: Evan Carter (LF)
  Spot 2: Marcus Semien (2B)
  Spot 3: Wyatt Langford (CF)
  Spot 4: Corey Seager (SS)
  Spot 5: Rowdy Tellez (1B)
  Spot 6: Josh Jung (3B)
  Spot 7: Leody Taveras (RF)
  Spot 8: Jonah Heim (C)
  Spot 9: Joc Pederson (DH)
Objective value: 0.7093390480331072

Termination vs LHP: optimal

Optimal lineup vs LHP:
  Spot 1: Josh Jung (3B)
  Spot 2: Joc Pederson (DH)
  Spot 3: Marcus Semien (2B)
  Spot 4: Wyatt Langford (CF)
  Spot 5: Corey Seager (SS)
  Spot 6: Rowdy Tellez (1B)
  Spot 7: Leody Taveras (RF)
  Spot 8: Jonah Heim (C)
  Spot 9: Evan Carter (LF)
Objective value: 0.7020376214258961
